# Tutorial: reglas SWRL con Owlready2 y GameOntology

En este tutorial nos centraremos **en cómo definir, ejecutar y consultar reglas SWRL desde Owlready2**.

En general:

> **Si una inferencia puede expresarse de forma natural mediante axiomas OWL/DL, no utilizaremos SWRL.**

Usaremos SWRL cuando sea útil **combinar relaciones, encadenar propiedades o realizar comparaciones sobre datos** de una forma que no resulte natural expresar con las construcciones DL vistas.

Trabajaremos sobre `GameOntologyv3.owl`.

## 1. Cargar la ontología

Coloca `GameOntologyv3.owl` en la misma carpeta que este notebook.

In [ ]:
from owlready2 import *
from pathlib import Path

archivo = "GameOntologyv3.owl"
onto = get_ontology(archivo)

try:
    onto.load()
    print("Ontología cargada correctamente")
except FileNotFoundError:
    print("No se encuentra el fichero", archivo)
except Exception as e:
    print(type(e).__name__, "-", e)

Ontología cargada correctamente


## 2. Reglas SWRL en Owlready2

En Owlready2 una regla SWRL se crea mediante `Imp()` y se define con `set_as_rule()`.

La forma general es:

```python
with onto:
    regla = Imp()
    regla.set_as_rule(
        "condicion1(?x), condicion2(?x, ?y) -> conclusion(?x, ?y)"
    )
```

Algunos detalles prácticos:

- Las condiciones del antecedente se separan mediante comas.
- Las variables comienzan por `?`.
- Las clases aparecen como `Clase(?x)`.
- Las propiedades de objeto aparecen como `propiedad(?x, ?y)`.
- Las propiedades de datos se utilizan de forma análoga.
- Crear la regla **no ejecuta** el razonamiento.
- Las reglas creadas pueden consultarse posteriormente mediante `onto.rules()`.

## 3. Primer ejemplo: encadenar dos relaciones

Queremos representar la relación `conocePersonaje(Player, Character)`:

> Un jugador conoce un personaje si juega a un juego que tiene ese personaje.

Utilizaremos las propiedades existentes `playsGame` y `hasCharacter`.

In [ ]:
with onto:
    class conocePersonaje(ObjectProperty):
        domain = [onto.Player]
        range = [onto.Character]

    regla_personajes = Imp()
    regla_personajes.set_as_rule(
        "playsGame(?p, ?g), hasCharacter(?g, ?c) -> conocePersonaje(?p, ?c)"
    )

print(regla_personajes)

playsGame(?p, ?g), hasCharacter(?g, ?c) -> conocePersonaje(?p, ?c)


Creamos algunos datos para comprobar la regla. Observa que **no añadimos explícitamente** ninguna relación `conocePersonaje`.

In [ ]:
ana = onto.Player("ana_swrl")
marta = onto.Player("marta_swrl")

juego1 = onto.Game("juego_swrl_1")
juego2 = onto.Game("juego_swrl_2")

mago = onto.Character("mago_swrl")
guerrera = onto.Character("guerrera_swrl")

ana.playsGame.append(juego1)
marta.playsGame.append(juego2)

juego1.hasCharacter.append(mago)
juego2.hasCharacter.append(guerrera)

print("Antes del razonamiento:")
print("Ana conoce:", ana.conocePersonaje)
print("Marta conoce:", marta.conocePersonaje)

Antes del razonamiento:
Ana conoce: []
Marta conoce: []


### Ejecutar Pellet

Para que Pellet materialice las relaciones inferidas, utilizamos `infer_property_values=True`.

In [ ]:
try:
    with onto:
        sync_reasoner_pellet(
            infer_property_values=True,
            infer_data_property_values=True
        )
    print("Razonamiento completado")
except OwlReadyInconsistentOntologyError:
    print("La ontología es inconsistente")
except Exception as e:
    print(type(e).__name__, "-", e)

* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jena-arq-fixed2.10.0.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jena-core-fixed2.10.0.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\je

* Owlready * Adding relation GameOntologyv3.marta_swrl conocePersonaje GameOntologyv3.guerrera_swrl
* Owlready * Adding relation GameOntologyv3.ana_swrl conocePersonaje GameOntologyv3.mago_swrl
Razonamiento completado


* Owlready2 * Pellet took 1.4280600547790527 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


In [ ]:
print("Después del razonamiento:")
print("Ana conoce:", ana.conocePersonaje)
print("Marta conoce:", marta.conocePersonaje)

Después del razonamiento:
Ana conoce: [GameOntologyv3.mago_swrl]
Marta conoce: [GameOntologyv3.guerrera_swrl]


## 4. Regla que combina tres relaciones

Podemos hacer una inferencia más específica.

Crearemos la propiedad `coincideConJugador`. Queremos inferir que dos jugadores coinciden cuando participan en una misma sesión.

Utilizaremos la propiedad existente `involvesPlayer`.

In [ ]:
with onto:
    class coincideConJugador(ObjectProperty):
        domain = [onto.Player]
        range = [onto.Player]

    regla_coinciden = Imp()
    regla_coinciden.set_as_rule(
        "Session(?s), involvesPlayer(?s, ?p1), involvesPlayer(?s, ?p2) "
        "-> coincideConJugador(?p1, ?p2)"
    )

print(regla_coinciden)

Session(?s), involvesPlayer(?s, ?p1), involvesPlayer(?s, ?p2) -> coincideConJugador(?p1, ?p2)


In [ ]:
sesion1 = onto.Session("sesion_swrl_1")

sesion1.involvesPlayer.append(ana)
sesion1.involvesPlayer.append(marta)

print("Antes:", ana.coincideConJugador)

with onto:
    sync_reasoner_pellet(infer_property_values=True)

print("Después:", ana.coincideConJugador)

Antes: []


* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jena-arq-fixed2.10.0.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jena-core-fixed2.10.0.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\je

* Owlready * Adding relation GameOntologyv3.marta_swrl coincideConJugador GameOntologyv3.marta_swrl
* Owlready * Adding relation GameOntologyv3.marta_swrl coincideConJugador GameOntologyv3.ana_swrl
* Owlready * Adding relation GameOntologyv3.ana_swrl coincideConJugador GameOntologyv3.marta_swrl
* Owlready * Adding relation GameOntologyv3.ana_swrl coincideConJugador GameOntologyv3.ana_swrl
Después: [GameOntologyv3.marta_swrl, GameOntologyv3.ana_swrl]


* Owlready2 * Pellet took 1.247408151626587 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


### Atención

La regla anterior también puede producir una relación de un jugador consigo mismo, porque SWRL no supone que `?p1` y `?p2` sean distintos.

Podemos exigir individuos diferentes utilizando `differentFrom`:

```python
regla.set_as_rule(
    "Session(?s), involvesPlayer(?s, ?p1), involvesPlayer(?s, ?p2), "
    "differentFrom(?p1, ?p2) -> coincideConJugador(?p1, ?p2)"
)
```

Para que esa condición pueda demostrarse, la diferencia entre los individuos debe estar disponible en la ontología.

Esto permite observar de nuevo que **nombres diferentes no implican automáticamente individuos diferentes en OWL**.

## 5. Reglas con propiedades de datos y built-ins

SWRL resulta especialmente útil para aplicar reglas de negocio sobre valores.

Añadiremos una puntuación numérica a los jugadores y una propiedad `nivelRecomendado`.

In [ ]:
with onto:
    class puntuacion(DataProperty, FunctionalProperty):
        domain = [onto.Player]
        range = [int]

    class nivelRecomendado(DataProperty, FunctionalProperty):
        domain = [onto.Player]
        range = [str]

ana.puntuacion = 1500
marta.puntuacion = 600

Los *built-ins* permiten realizar comparaciones. En Owlready2 pueden escribirse directamente en el texto de la regla.

Por ejemplo, recomendaremos el nivel `"avanzado"` a jugadores con más de 1000 puntos.

In [ ]:
with onto:
    regla_nivel = Imp()
    regla_nivel.set_as_rule(
        'Player(?p), puntuacion(?p, ?x), greaterThan(?x, 1000) '
        '-> nivelRecomendado(?p, "avanzado")'
    )

print(regla_nivel)

Player(?p), puntuacion(?p, ?x), greaterThan(?x, 1000) -> nivelRecomendado(?p, 'avanzado')


In [ ]:
print("Antes:")
print("Ana:", ana.nivelRecomendado)
print("Marta:", marta.nivelRecomendado)

with onto:
    sync_reasoner_pellet(
        infer_property_values=True,
        infer_data_property_values=True
    )

print("\nDespués:")
print("Ana:", ana.nivelRecomendado)
print("Marta:", marta.nivelRecomendado)

Antes:
Ana: None
Marta: None


* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jena-arq-fixed2.10.0.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jena-core-fixed2.10.0.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\je

* Owlready * Adding relation GameOntologyv3.ana_swrl nivelRecomendado avanzado

Después:
Ana: avanzado
Marta: None


* Owlready2 * Pellet took 1.3262410163879395 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


Otros *built-ins* útiles incluyen comparaciones como:

- `greaterThan(?x, valor)`
- `greaterThanOrEqual(?x, valor)`
- `lessThan(?x, valor)`
- `lessThanOrEqual(?x, valor)`
- `equal(?x, ?y)`
- `notEqual(?x, ?y)`

## 6. Una regla que combina relaciones y datos

Podemos combinar propiedades de objeto y propiedades de datos.

Crearemos `recomiendaJuego(Player, Game)`.

La regla recomendará un juego a un jugador cuando:

- el jugador juegue a ese juego;
- el jugador tenga más de 1000 puntos.

El objetivo aquí es observar la sintaxis Owlready2 de una regla que mezcla distintos tipos de átomos.

In [ ]:
with onto:
    class recomiendaJuego(ObjectProperty):
        domain = [onto.Player]
        range = [onto.Game]

    regla_recomendacion = Imp()
    regla_recomendacion.set_as_rule(
        "playsGame(?p, ?g), puntuacion(?p, ?x), greaterThan(?x, 1000) "
        "-> recomiendaJuego(?p, ?g)"
    )

print(regla_recomendacion)

playsGame(?p, ?g), puntuacion(?p, ?x), greaterThan(?x, 1000) -> recomiendaJuego(?p, ?g)


In [ ]:
print("Antes:", ana.recomiendaJuego)

with onto:
    sync_reasoner_pellet(
        infer_property_values=True,
        infer_data_property_values=True
    )

print("Después:", ana.recomiendaJuego)

Antes: []


* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jena-arq-fixed2.10.0.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\jena-core-fixed2.10.0.jar;C:\Users\Laura\PyCharmMiscProject\.venv\Lib\site-packages\owlready2\pellet\je

* Owlready * Adding relation GameOntologyv3.ana_swrl recomiendaJuego GameOntologyv3.juego_swrl_1
Después: [GameOntologyv3.juego_swrl_1]


* Owlready2 * Pellet took 1.260615348815918 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


## 7. Consultar y modificar reglas

Las reglas SWRL forman parte de la ontología.

Podemos listarlas:

In [ ]:
for regla in onto.rules():
    print(regla)

playsGame(?p, ?g), puntuacion(?p, ?x), greaterThan(?x, 1000) -> recomiendaJuego(?p, ?g)
Player(?p), puntuacion(?p, ?x), greaterThan(?x, 1000) -> nivelRecomendado(?p, 'avanzado')
Session(?s), involvesPlayer(?s, ?p1), involvesPlayer(?s, ?p2) -> coincideConJugador(?p1, ?p2)
playsGame(?p, ?g), hasCharacter(?g, ?c) -> conocePersonaje(?p, ?c)


También podemos eliminar una regla cuando solo se ha creado para realizar un experimento:

In [ ]:
# Ejemplo:
# destroy_entity(regla_recomendacion)
#
# Después:
# for regla in onto.rules():
#     print(regla)

Esto es útil en un notebook cuando queremos probar reglas diferentes sin acumular reglas que puedan afectar a experimentos posteriores.

# Ejercicios

Para cada ejercicio:

1. identifica qué información necesita la regla;
2. crea únicamente las clases o propiedades adicionales necesarias;
3. escribe la regla con `Imp()` y `set_as_rule()`;
4. crea datos de prueba;
5. muestra el resultado antes de Pellet;
6. ejecuta Pellet;
7. muestra y explica la inferencia obtenida.

Evita introducir explícitamente la conclusión que debe obtener la regla.

## Ejercicio 1 — Personajes disponibles para un jugador

Crea una propiedad:

`puedeUsarPersonaje(Player, Character)`

Un jugador puede usar un personaje cuando juega a un juego que contiene ese personaje.

Utiliza `playsGame` y `hasCharacter`.

Crea al menos dos juegos, dos jugadores y tres personajes de forma que sea posible comprobar que la regla no relaciona a un jugador con personajes pertenecientes a juegos que no juega.

In [ ]:
# Tu solución

## Ejercicio 2 — Jugadores que han coincidido

Crea una propiedad:

`haCoincididoCon(Player, Player)`

Dos jugadores han coincidido cuando aparecen como participantes de una misma `Session`.

Evita inferir que un jugador ha coincidido consigo mismo.

Crea al menos dos sesiones y tres jugadores para probar la regla. Comprueba qué relaciones se obtienen después de Pellet.

**Pregunta:** ¿qué información debe conocer la ontología para poder demostrar que dos jugadores son distintos?

In [ ]:
# Tu solución

## Ejercicio 3 — Recomendación basada en datos

Crea:

- una propiedad de datos funcional `horasJugadas` para `Player`;
- una propiedad de objeto `recomiendaJuego` entre `Player` y `Game`.

Diseña una regla que recomiende un juego a un jugador cuando:

- el jugador ya juega a ese juego;
- tiene al menos 500 horas jugadas.

Utiliza un *built-in* de comparación.

Crea tres jugadores con distintos valores de `horasJugadas` y comprueba las recomendaciones obtenidas.

In [ ]:
# Tu solución

## Ejercicio 4 — Propagar información a través de varias relaciones

Crea una propiedad:

`conoceLogro(Player, Achievement)`

Queremos inferir que un jugador conoce un logro cuando:

1. juega a un juego;
2. ese juego tiene dicho logro.

Utiliza las propiedades existentes de GameOntology para construir la regla.

Crea datos de prueba suficientes para comprobar que un logro perteneciente a otro juego no se asigna al jugador.

In [ ]:
# Tu solución